In [ ]:
pip install -q streamlit pyjwt bcrypt plotly streamlit-option-menu

In [8]:
%%writefile app.py

import os
import re
import sqlite3
import base64
import jwt
import bcrypt
import datetime
import time
import secrets
import hashlib
import smtplib
import hmac
import traceback

import streamlit as st
import plotly.graph_objects as go

from streamlit_option_menu import option_menu
from email.mime.text import MIMEText


# ============================================================
# STREAMLIT CONFIGURATION
# ============================================================

os.makedirs(".streamlit", exist_ok=True)

with open(".streamlit/config.toml", "w") as f:
    f.write(
        '[theme]\n'
        'base="light"\n'
        'primaryColor="#ffd803"\n'
        'backgroundColor="#f9fcfc"\n'
        'secondaryBackgroundColor="#e3f6f5"\n'
        'textColor="#2d334a"\n'
    )

st.set_page_config(
    page_title="Maritime AI",
    page_icon="🚢",
    layout="wide",
    initial_sidebar_state="expanded"
)


# ============================================================
# SECRETS: ENVIRONMENT VARIABLES + GOOGLE COLAB SECRETS
# ============================================================

def get_secret(name):
    value = os.environ.get(name)

    if value:
        return value.strip()

    try:
        from google.colab import userdata
        value = userdata.get(name)

        if value:
            return value.strip()

    except Exception:
        pass

    return None


JWT_SECRET = get_secret("JWT_SECRET")

if not JWT_SECRET:
    st.error(
        "JWT_SECRET is missing. Add JWT_SECRET to Google Colab "
        "Secrets, enable notebook access, and load it into the "
        "environment before starting Streamlit."
    )
    st.stop()


# ============================================================
# COLORS
# ============================================================

COLORS = {
    "bg_main": "#f9fcfc",
    "bg_sidebar": "#e3f6f5",
    "bg_card": "#ffffff",
    "bg_card_alt": "#bae8e8",
    "text_main": "#2d334a",
    "text_heading": "#272343",
    "text_muted": "#64748b",
    "accent": "#ffd803",
    "accent_hover": "#e6c300",
    "accent_text": "#272343",
    "border": "#272343",
    "border_light": "#bae8e8",
    "success": "#34d399",
    "danger": "#f87171"
}

# ============================================================
# MARITIME BACKGROUND IMAGE
# ============================================================

BACKGROUND_IMAGE_PATH = "maritime_background.png"

with open(BACKGROUND_IMAGE_PATH, "rb") as image_file:
    background_base64 = base64.b64encode(
        image_file.read()
    ).decode("utf-8")
# ============================================================
# CUSTOM CSS + ANIMATED MARITIME BACKGROUND
# ============================================================

st.markdown(
    f"""
<style>
    @import url(
        'https://fonts.googleapis.com/css2?family=Poppins:wght@400;500;600;700&family=Inter:wght@300;400;500;600&display=swap'
    );

    html, body {{
    background: transparent !important;
    font-family: 'Inter', sans-serif !important;
    color: {COLORS['text_main']} !important;
}}

.stApp {{
    background:
        linear-gradient(
            rgba(249, 252, 252, 0.18),
            rgba(249, 252, 252, 0.18)
        ),
        url("data:image/png;base64,{background_base64}")
        center center / cover no-repeat fixed !important;

    font-family: 'Inter', sans-serif !important;
    color: {COLORS['text_main']} !important;
}}

    footer, div[data-testid="stDecoration"] {{
        visibility: hidden !important;
        display: none !important;
    }}

    header {{
        background: transparent !important;
        z-index: 999999 !important;
    }}

    button[kind="header"],
    div[data-testid="stSidebarCollapsedControl"] button {{
        visibility: visible !important;
        display: flex !important;
        opacity: 1 !important;
        background-color: {COLORS['accent']} !important;
        border: 2px solid {COLORS['border']} !important;
        border-radius: 8px !important;
        padding: 6px !important;
        margin: 8px !important;
        box-shadow: 3px 3px 0px {COLORS['border']} !important;
    }}

    button[kind="header"] svg,
    div[data-testid="stSidebarCollapsedControl"] svg {{
        fill: {COLORS['text_heading']} !important;
        color: {COLORS['text_heading']} !important;
        stroke: {COLORS['text_heading']} !important;
    }}

    .block-container {{
        padding: 2rem 2.5rem !important;
        max-width: 1200px;
        position: relative;
        z-index: 2;
    }}

    h1, h2, h3, h4 {{
        font-family: 'Poppins', sans-serif !important;
        color: {COLORS['text_heading']} !important;
    }}

    label p {{
        font-weight: 600 !important;
        color: {COLORS['text_heading']} !important;
    }}

    div[data-baseweb="base-input"],
    div[data-baseweb="select"] > div {{
        background-color: transparent !important;
        border: none !important;
    }}

    div[data-baseweb="input"],
    div[data-baseweb="select"] {{
        background-color: {COLORS['bg_card']} !important;
        border: 2px solid {COLORS['border']} !important;
        border-radius: 10px !important;
    }}

    div[data-baseweb="input"]:focus-within {{
        border-color: {COLORS['accent']} !important;
        box-shadow: 4px 4px 0px {COLORS['border']} !important;
    }}

    input, div[data-baseweb="select"] span {{
        color: {COLORS['text_main']} !important;
        -webkit-text-fill-color: {COLORS['text_main']} !important;
    }}

    div[data-testid="stButton"] button {{
        background-color: {COLORS['accent']} !important;
        color: {COLORS['accent_text']} !important;
        border: 2px solid {COLORS['border']} !important;
        border-radius: 10px !important;
        font-family: 'Inter', sans-serif !important;
        font-weight: 700 !important;
        font-size: 14px !important;
        height: 48px !important;
        min-height: 48px !important;
        white-space: nowrap !important;
        display: flex !important;
        align-items: center !important;
        justify-content: center !important;
        padding: 0px 16px !important;
        box-shadow: 4px 4px 0px {COLORS['border']} !important;
        width: 100%;
        transition: all 0.2s ease !important;
    }}

    div[data-testid="stButton"] button:hover {{
        background-color: {COLORS['accent_hover']} !important;
        transform: translate(-2px, -2px) !important;
        box-shadow: 6px 6px 0px {COLORS['border']} !important;
    }}

    section[data-testid="stSidebar"] {{
        background: {COLORS['bg_sidebar']} !important;
        border-right: 2px solid {COLORS['border']} !important;
        z-index: 3 !important;
    }}

    .pn-card {{
        background: {COLORS['bg_card']};
        border: 2px solid {COLORS['border']};
        border-radius: 14px;
        padding: 24px;
        box-shadow: 4px 4px 0px {COLORS['border_light']};
    }}

    /* Subtle animated ships in the background */
    .maritime-animation {{
        position: fixed;
        inset: 0;
        width: 100%;
        height: 100%;
        overflow: hidden;
        pointer-events: none;
        z-index: 0;
        opacity: 0.10;
    }}

    .maritime-ship {{
        position: absolute;
        left: -100px;
        white-space: nowrap;
        animation-name: ship-sail;
        animation-timing-function: linear;
        animation-iteration-count: infinite;
        pointer-events: none;
        user-select: none;
    }}

    .ship-one {{
        top: 17%;
        font-size: 29px;
        animation-duration: 55s;
    }}

    .ship-two {{
        top: 48%;
        font-size: 22px;
        animation-duration: 72s;
        animation-delay: -25s;
    }}

    .ship-three {{
        top: 78%;
        font-size: 25px;
        animation-duration: 63s;
        animation-delay: -40s;
    }}

    @keyframes ship-sail {{
        from {{
            transform: translateX(-120px);
        }}
        to {{
            transform: translateX(calc(100vw + 150px));
        }}
    }}

    @media (prefers-reduced-motion: reduce) {{
        .maritime-ship {{
            animation: none !important;
            display: none !important;
        }}
    }}
</style>
""",
    unsafe_allow_html=True
)
# ============================================================
# CONSTANTS
# ============================================================

LOCKOUT_SECONDS = 20 * 60
MAX_LOGIN_ATTEMPTS = 5
MAX_OTP_ATTEMPTS = 5
OTP_VALIDITY_SECONDS = 300


# ============================================================
# DATABASE AND PASSWORD FUNCTIONS
# ============================================================

def get_db():
    return sqlite3.connect(
        "infosys_portal.db",
        check_same_thread=False,
        timeout=30
    )


def hash_txt(text):
    return bcrypt.hashpw(
        text.encode("utf-8"),
        bcrypt.gensalt()
    ).decode("utf-8")


def check_txt(text, hashed_text):
    if not hashed_text:
        return False

    try:
        return bcrypt.checkpw(
            text.encode("utf-8"),
            hashed_text.encode("utf-8")
        )
    except (ValueError, TypeError):
        return False


# ============================================================
# PASSWORD VALIDATION
# ============================================================

def is_valid_password(password):
    return (
        bool(password)
        and len(password) >= 8
        and not any(char.isspace() for char in password)
    )


def password_matches_current(email, new_password):
    """
    Compare the proposed password with the current stored bcrypt hash.
    The plaintext password is never stored in the database.
    """

    with get_db() as conn:
        result = conn.execute(
            """
            SELECT password_hash
            FROM users
            WHERE email = ?
            """,
            (email,)
        ).fetchone()

    if not result or not result[0]:
        return False

    return check_txt(new_password, result[0])


# ============================================================
# EMAIL VALIDATION: .COM ONLY
# ============================================================

def is_valid_email(email):
    email = email.strip().lower()

    pattern = (
        r"^[a-zA-Z0-9._%+-]+@"
        r"[a-zA-Z0-9-]+(?:\.[a-zA-Z0-9-]+)*"
        r"\.com$"
    )

    if not re.fullmatch(pattern, email):
        return False

    if ".." in email:
        return False

    local, domain = email.rsplit("@", 1)

    if not local or not domain:
        return False

    if local.startswith(".") or local.endswith("."):
        return False

    labels = domain.split(".")

    if any(
        not label
        or label.startswith("-")
        or label.endswith("-")
        for label in labels
    ):
        return False

    return True


# ============================================================
# GMAIL OTP FUNCTION
# ============================================================

def send_otp_email(receiver_email, otp):
    sender_email = get_secret("GMAIL_ADDRESS")
    app_password = get_secret("EMAIL_PASSWORD")

    if not sender_email or not app_password:
        raise ValueError(
            "Gmail credentials are missing. Check GMAIL_ADDRESS "
            "and EMAIL_PASSWORD in Google Colab Secrets."
        )

    message = MIMEText(
        f"""
Hello,

You requested to reset your Maritime AI Portal password.

Your OTP is: {otp}

This OTP is valid for 5 minutes.

If you did not request this password reset, please ignore this email.

Regards,
Maritime AI Portal
        """.strip()
    )

    message["Subject"] = "Maritime AI - Password Reset OTP"
    message["From"] = sender_email
    message["To"] = receiver_email

    with smtplib.SMTP(
        "smtp.gmail.com",
        587,
        timeout=20
    ) as server:
        server.ehlo()
        server.starttls()
        server.ehlo()
        server.login(sender_email, app_password)
        server.send_message(message)


# ============================================================
# DATABASE INITIALIZATION
# ============================================================

with get_db() as conn:

    conn.execute(
        """
        CREATE TABLE IF NOT EXISTS users (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            username TEXT UNIQUE,
            email TEXT UNIQUE,
            password_hash TEXT,
            security_question TEXT,
            security_answer_hash TEXT
        )
        """
    )

    conn.execute(
        """
        CREATE TABLE IF NOT EXISTS login_attempts (
            email TEXT PRIMARY KEY,
            failed_attempts INTEGER NOT NULL DEFAULT 0,
            locked_until REAL NOT NULL DEFAULT 0
        )
        """
    )

    conn.execute(
        """
        CREATE TABLE IF NOT EXISTS otp_attempts (
            email TEXT PRIMARY KEY,
            failed_attempts INTEGER NOT NULL DEFAULT 0,
            locked_until REAL NOT NULL DEFAULT 0
        )
        """
    )

    # DEMO ADMIN ACCOUNT:
    # Change this setup before making the application public.
    admin_exists = conn.execute(
        "SELECT id FROM users WHERE email = ?",
        ("infosys@ai",)
    ).fetchone()

    if not admin_exists:
        conn.execute(
            """
            INSERT INTO users
                (username, email, password_hash,
                 security_question, security_answer_hash)
            VALUES (?, ?, ?, ?, ?)
            """,
            (
                "Administrator",
                "infosys@ai",
                hash_txt("admin@123"),
                "What is your pet name?",
                hash_txt("admin")
            )
        )


# ============================================================
# LOGIN LOCKOUT FUNCTIONS
# ============================================================

def get_login_lock_status(email):
    with get_db() as conn:
        row = conn.execute(
            """
            SELECT failed_attempts, locked_until
            FROM login_attempts
            WHERE email = ?
            """,
            (email,)
        ).fetchone()

    if row and row[1] > time.time():
        return True, int(row[1] - time.time()) + 1

    return False, 0


def get_failed_login_count(email):
    with get_db() as conn:
        row = conn.execute(
            """
            SELECT failed_attempts
            FROM login_attempts
            WHERE email = ?
            """,
            (email,)
        ).fetchone()

    return row[0] if row else 0


def record_failed_login(email):
    now = time.time()

    with get_db() as conn:
        row = conn.execute(
            """
            SELECT failed_attempts, locked_until
            FROM login_attempts
            WHERE email = ?
            """,
            (email,)
        ).fetchone()

        if row and row[1] > now:
            return

        attempts = row[0] if row and row[1] == 0 else 0

        if row and row[1] > 0 and row[1] <= now:
            attempts = 0

        attempts += 1

        locked_until = (
            now + LOCKOUT_SECONDS
            if attempts >= MAX_LOGIN_ATTEMPTS
            else 0
        )

        conn.execute(
            """
            INSERT INTO login_attempts
                (email, failed_attempts, locked_until)
            VALUES (?, ?, ?)
            ON CONFLICT(email) DO UPDATE SET
                failed_attempts = excluded.failed_attempts,
                locked_until = excluded.locked_until
            """,
            (email, attempts, locked_until)
        )


def clear_failed_logins(email):
    with get_db() as conn:
        conn.execute(
            "DELETE FROM login_attempts WHERE email = ?",
            (email,)
        )


# ============================================================
# OTP LOCKOUT FUNCTIONS
# ============================================================

def get_otp_lock_status(email):
    with get_db() as conn:
        row = conn.execute(
            """
            SELECT failed_attempts, locked_until
            FROM otp_attempts
            WHERE email = ?
            """,
            (email,)
        ).fetchone()

    if row and row[1] > time.time():
        return True, int(row[1] - time.time()) + 1

    return False, 0


def get_otp_failed_count(email):
    with get_db() as conn:
        row = conn.execute(
            """
            SELECT failed_attempts
            FROM otp_attempts
            WHERE email = ?
            """,
            (email,)
        ).fetchone()

    return row[0] if row else 0


def record_failed_otp(email):
    now = time.time()

    with get_db() as conn:
        row = conn.execute(
            """
            SELECT failed_attempts, locked_until
            FROM otp_attempts
            WHERE email = ?
            """,
            (email,)
        ).fetchone()

        if row and row[1] > now:
            return

        attempts = row[0] if row and row[1] == 0 else 0

        if row and row[1] > 0 and row[1] <= now:
            attempts = 0

        attempts += 1

        locked_until = (
            now + LOCKOUT_SECONDS
            if attempts >= MAX_OTP_ATTEMPTS
            else 0
        )

        conn.execute(
            """
            INSERT INTO otp_attempts
                (email, failed_attempts, locked_until)
            VALUES (?, ?, ?)
            ON CONFLICT(email) DO UPDATE SET
                failed_attempts = excluded.failed_attempts,
                locked_until = excluded.locked_until
            """,
            (email, attempts, locked_until)
        )


def clear_otp_attempts(email):
    with get_db() as conn:
        conn.execute(
            "DELETE FROM otp_attempts WHERE email = ?",
            (email,)
        )


# ============================================================
# JWT FUNCTIONS
# ============================================================

def make_jwt(email):
    return jwt.encode(
        {
            "email": email,
            "exp": datetime.datetime.now(
                datetime.timezone.utc
            ) + datetime.timedelta(hours=2)
        },
        JWT_SECRET,
        algorithm="HS256"
    )


def verify_jwt(token):
    try:
        return jwt.decode(
            token,
            JWT_SECRET,
            algorithms=["HS256"]
        )
    except jwt.InvalidTokenError:
        return None


# ============================================================
# SESSION STATE
# ============================================================

default_values = {
    "token": None,
    "page": "Login",
    "reset_email": None,
    "reset_mode": None,
    "otp_hash": None,
    "otp_expiry": None,
    "otp_sent": False,
    "otp_verified": False,
    "otp_attempts": 0,
    "sq_p": None
}

for key, value in default_values.items():
    if key not in st.session_state:
        st.session_state[key] = value


def navigate(page):
    st.session_state.page = page
    st.rerun()


def clear_reset_state():
    st.session_state.reset_email = None
    st.session_state.reset_mode = None
    st.session_state.otp_hash = None
    st.session_state.otp_expiry = None
    st.session_state.otp_sent = False
    st.session_state.otp_verified = False
    st.session_state.otp_attempts = 0
    st.session_state.sq_p = None


# ============================================================
# AUTHENTICATION HEADER
# ============================================================

def auth_header(
    title,
    sub="Agentic AI for Maritime Freight Pricing and Route Optimization"
):
    st.markdown(
        f"""
        <div style="text-align:center;padding:1.5rem 0 1rem;">
            <div style="font-size:40px;margin-bottom:10px;">🚢</div>
            <h1 style="font-size:2rem !important;margin:0;">
                Maritime AI
            </h1>
            <p style="color:{COLORS['text_muted']};font-size:14px;margin:4px 0 0;">
                {sub}
            </p>
        </div>

        <div style="text-align:center;margin-bottom:1.5rem;">
            <span style="font-size:1.1rem;font-weight:700;color:{COLORS['text_heading']};">
                {title}
            </span>
        </div>
        """,
        unsafe_allow_html=True
    )


# ============================================================
# PAGE ROUTING: LOGIN / SIGNUP / FORGOT PASSWORD
# ============================================================

if not st.session_state.token:

    if st.session_state.page not in ["Login", "Signup", "Forgot"]:
        st.session_state.page = "Login"

    _, mid, _ = st.columns([1, 1.45, 1])

    with mid:

        # ====================================================
        # LOGIN PAGE
        # ====================================================

        if st.session_state.page == "Login":

            auth_header("Sign in to your account")

            email = st.text_input(
                "Email address",
                placeholder="you@gmail.com",
                key="login_email"
            ).lower().strip()

            password = st.text_input(
                "Password",
                type="password",
                placeholder="Enter your password",
                key="login_password"
            )

            st.markdown("<br>", unsafe_allow_html=True)

            col_l, col_c, col_r = st.columns([1, 1.15, 1.3])

            if col_l.button("Sign In →", use_container_width=True):

                if not email or not password:
                    st.error("Please enter your email and password.")

                else:
                    is_locked, remaining = get_login_lock_status(email)

                    if is_locked:
                        st.error(
                            "🔒 Account temporarily locked. "
                            f"Try again in {remaining // 60}m "
                            f"{remaining % 60}s."
                        )

                    else:
                        with get_db() as conn:
                            result = conn.execute(
                                """
                                SELECT password_hash
                                FROM users
                                WHERE email = ?
                                """,
                                (email,)
                            ).fetchone()

                        if result and check_txt(password, result[0]):
                            clear_failed_logins(email)

                            st.session_state.token = make_jwt(email)
                            navigate("Dashboard")

                        else:
                            record_failed_login(email)

                            is_locked, remaining = get_login_lock_status(email)

                            if is_locked:
                                st.error(
                                    "🔒 Five incorrect login attempts. "
                                    "Access is locked for 20 minutes."
                                )
                            else:
                                attempts_left = (
                                    MAX_LOGIN_ATTEMPTS
                                    - get_failed_login_count(email)
                                )

                                st.error(
                                    "❌ Invalid credentials. "
                                    f"{attempts_left} attempts remaining."
                                )

            if col_c.button("Create Account", use_container_width=True):
                navigate("Signup")

            if col_r.button("Forgot Password", use_container_width=True):
                navigate("Forgot")


        # ====================================================
        # SIGNUP PAGE
        # ====================================================

        elif st.session_state.page == "Signup":

            auth_header("Create an account")

            username = st.text_input(
                "Full name / Username",
                placeholder="Jane Doe",
                key="signup_username"
            ).strip()

            email = st.text_input(
                "Email address",
                placeholder="you@gmail.com",
                key="signup_email"
            ).lower().strip()

            password = st.text_input(
                "Password",
                type="password",
                key="signup_password"
            )

            confirm_password = st.text_input(
                "Confirm password",
                type="password",
                key="signup_confirm_password"
            )

            security_question = st.selectbox(
                "Security Question",
                [
                    "What is your pet name?",
                    "What is your mother's maiden name?",
                    "What is your favourite city?"
                ]
            )

            security_answer = st.text_input(
                "Your answer",
                key="signup_security_answer"
            ).strip()

            st.caption(
                "Use at least 8 characters and do not include whitespace."
            )

            st.markdown("<br>", unsafe_allow_html=True)

            if st.button(
                "Create Account & Login →",
                use_container_width=True
            ):

                if (
                    not username
                    or not email
                    or not password
                    or not security_answer
                ):
                    st.error("Please fill in all fields.")

                elif not is_valid_email(email):
                    st.error(
                        "Invalid email address. Only valid email "
                        "addresses ending in .com are accepted."
                    )

                elif not is_valid_password(password):
                    st.error(
                        "Password must contain at least 8 characters "
                        "and must not contain whitespace."
                    )

                elif password != confirm_password:
                    st.error("Passwords do not match.")

                else:
                    try:
                        with get_db() as conn:
                            conn.execute(
                                """
                                INSERT INTO users
                                    (username, email, password_hash,
                                     security_question, security_answer_hash)
                                VALUES (?, ?, ?, ?, ?)
                                """,
                                (
                                    username,
                                    email,
                                    hash_txt(password),
                                    security_question,
                                    hash_txt(security_answer.lower().strip())
                                )
                            )

                        clear_failed_logins(email)
                        st.session_state.token = make_jwt(email)

                        st.success("Account created successfully!")
                        time.sleep(1)
                        navigate("Dashboard")

                    except sqlite3.IntegrityError:
                        st.error("Email or username already registered.")

            st.markdown("<br>", unsafe_allow_html=True)

            if st.button("← Back to Sign In", use_container_width=True):
                navigate("Login")


        # ====================================================
        # FORGOT PASSWORD PAGE
        # ====================================================

        elif st.session_state.page == "Forgot":

            auth_header(
                "Reset your password",
                "Choose your verification method"
            )

            # ------------------------------------------------
            # STEP 1: ENTER REGISTERED EMAIL
            # ------------------------------------------------

            if not st.session_state.reset_email:

                email = st.text_input(
                    "Registered email address",
                    placeholder="you@gmail.com",
                    key="forgot_registered_email"
                ).lower().strip()

                st.markdown("<br>", unsafe_allow_html=True)

                col_sq, col_otp = st.columns(2)

                if col_sq.button(
                    "Via Security Question",
                    use_container_width=True
                ):

                    if not email:
                        st.error("Please enter your registered email.")

                    else:
                        with get_db() as conn:
                            result = conn.execute(
                                """
                                SELECT security_question
                                FROM users
                                WHERE email = ?
                                """,
                                (email,)
                            ).fetchone()

                        if result:
                            st.session_state.reset_email = email
                            st.session_state.sq_p = result[0]
                            st.session_state.reset_mode = "sq"
                            st.rerun()
                        else:
                            st.error("Email address not found.")

                if col_otp.button(
                    "Via OTP",
                    use_container_width=True
                ):

                    if not email:
                        st.error("Please enter your registered email.")

                    else:
                        with get_db() as conn:
                            result = conn.execute(
                                "SELECT id FROM users WHERE email = ?",
                                (email,)
                            ).fetchone()

                        if result:
                            is_locked, remaining = get_otp_lock_status(email)

                            if is_locked:
                                st.error(
                                    "🔒 OTP verification is locked. "
                                    f"Try again in {remaining // 60}m "
                                    f"{remaining % 60}s."
                                )
                            else:
                                st.session_state.reset_email = email
                                st.session_state.reset_mode = "otp"
                                st.session_state.otp_hash = None
                                st.session_state.otp_expiry = None
                                st.session_state.otp_sent = False
                                st.session_state.otp_verified = False
                                st.session_state.otp_attempts = 0
                                st.rerun()
                        else:
                            st.error("Email address not found.")


            # ------------------------------------------------
            # STEP 2: SECURITY QUESTION VERIFICATION
            # ------------------------------------------------

            elif st.session_state.reset_mode == "sq":

                st.info(
                    f"Security Question: {st.session_state.sq_p}"
                )

                answer = st.text_input(
                    "Your answer",
                    key="security_answer"
                ).lower().strip()

                new_password = st.text_input(
                    "New password (minimum 8 characters)",
                    type="password",
                    key="sq_new_password"
                )

                confirm_password = st.text_input(
                    "Confirm new password",
                    type="password",
                    key="sq_confirm_password"
                )

                if st.button("Reset Password →", use_container_width=True):

                    if not is_valid_password(new_password):
                        st.error(
                            "Password must contain at least 8 characters "
                            "and must not contain whitespace."
                        )

                    elif new_password != confirm_password:
                        st.error("Passwords do not match.")

                    else:
                        with get_db() as conn:
                            result = conn.execute(
                                """
                                SELECT security_answer_hash
                                FROM users
                                WHERE email = ?
                                """,
                                (st.session_state.reset_email,)
                            ).fetchone()

                        if result and check_txt(answer, result[0]):

                            # NEW SECURITY CHECK:
                            # Do not allow the current password to be reused.
                            if password_matches_current(
                                st.session_state.reset_email,
                                new_password
                            ):
                                st.error(
                                    "You cannot reuse your current password. "
                                    "Please choose a different password."
                                )

                            else:
                                with get_db() as conn:
                                    cursor = conn.execute(
                                        """
                                        UPDATE users
                                        SET password_hash = ?
                                        WHERE email = ?
                                        """,
                                        (
                                            hash_txt(new_password),
                                            st.session_state.reset_email
                                        )
                                    )

                                if cursor.rowcount == 1:
                                    clear_reset_state()
                                    st.success(
                                        "Password updated successfully!"
                                    )
                                    time.sleep(1)
                                    navigate("Login")
                                else:
                                    st.error(
                                        "Password could not be updated. "
                                        "Please try again."

                                    )

                        else:
                            st.error("Incorrect security answer.")


            # ------------------------------------------------
            # STEP 3: GMAIL OTP FLOW
            # ------------------------------------------------

            elif st.session_state.reset_mode == "otp":

                reset_email = st.session_state.reset_email

                st.info(f"Password reset email: {reset_email}")

                # --------------------------------------------
                # SEND OTP
                # --------------------------------------------

                if not st.session_state.otp_sent:

                    is_locked, remaining = get_otp_lock_status(reset_email)

                    if is_locked:
                        st.error(
                            "🔒 OTP verification is locked. "
                            f"Try again in {remaining // 60}m "
                            f"{remaining % 60}s."
                        )

                    else:
                        st.write(
                            "Click below to receive a 6-digit OTP "
                            "in your registered email inbox."
                        )

                        if st.button(
                            "Send OTP to Email",
                            use_container_width=True
                        ):

                            is_locked, remaining = get_otp_lock_status(
                                reset_email
                            )

                            if is_locked:
                                st.error(
                                    "OTP sending is blocked during "
                                    "the 20-minute lockout."
                                )

                            else:
                                try:
                                    otp = f"{secrets.randbelow(1000000):06d}"

                                    send_otp_email(reset_email, otp)

                                    st.session_state.otp_hash = (
                                        hashlib.sha256(
                                            otp.encode("utf-8")
                                        ).hexdigest()
                                    )

                                    st.session_state.otp_expiry = (
                                        time.time() + OTP_VALIDITY_SECONDS
                                    )

                                    st.session_state.otp_sent = True
                                    st.session_state.otp_verified = False
                                    st.session_state.otp_attempts = 0

                                    st.success(
                                        "OTP sent successfully! "
                                        "Check your inbox and spam folder."
                                    )

                                    st.rerun()

                                except Exception:
                                    st.error(
                                        "Unable to send OTP. "
                                        "Check your Gmail configuration "
                                        "and try again."
                                    )

                                    print(
                                        "OTP email sending failed:\n",
                                        traceback.format_exc()
                                    )


                # --------------------------------------------
                # VERIFY OTP
                # --------------------------------------------

                elif not st.session_state.otp_verified:

                    is_locked, remaining = get_otp_lock_status(reset_email)

                    if is_locked:
                        st.error(
                            "🔒 OTP verification is locked. "
                            f"Try again in {remaining // 60}m "
                            f"{remaining % 60}s."
                        )

                    else:
                        entered_otp = st.text_input(
                            "Enter the 6-digit OTP",
                            max_chars=6,
                            key="entered_otp"
                        ).strip()

                        if st.button("Verify OTP", use_container_width=True):

                            is_locked, remaining = get_otp_lock_status(
                                reset_email
                            )

                            if is_locked:
                                st.error(
                                    "OTP verification is locked. "
                                    f"Try again in {remaining // 60}m "
                                    f"{remaining % 60}s."
                                )

                            elif (
                                st.session_state.otp_expiry is None
                                or time.time() > st.session_state.otp_expiry
                            ):
                                st.session_state.otp_hash = None
                                st.session_state.otp_expiry = None
                                st.session_state.otp_sent = False
                                st.session_state.otp_verified = False

                                st.error(
                                    "OTP expired. Please request a new OTP."
                                )

                            elif not (
                                len(entered_otp) == 6
                                and entered_otp.isdigit()
                            ):
                                record_failed_otp(reset_email)

                                is_locked, remaining = get_otp_lock_status(
                                    reset_email
                                )

                                if is_locked:
                                    st.session_state.otp_hash = None
                                    st.session_state.otp_expiry = None
                                    st.session_state.otp_sent = False
                                    st.session_state.otp_verified = False

                                    st.error(
                                        "🔒 Five incorrect OTP attempts. "
                                        "Verification is locked for 20 minutes."
                                    )
                                else:
                                    attempts_left = (
                                        MAX_OTP_ATTEMPTS
                                        - get_otp_failed_count(reset_email)
                                    )
                                    st.error(
                                        "Enter a valid 6-digit OTP. "
                                        f"{attempts_left} attempts remaining."
                                    )

                            else:
                                entered_hash = hashlib.sha256(
                                    entered_otp.encode("utf-8")
                                ).hexdigest()

                                expected_hash = (
                                    st.session_state.otp_hash or ""
                                )

                                if hmac.compare_digest(
                                    entered_hash,
                                    expected_hash
                                ):
                                    clear_otp_attempts(reset_email)

                                    st.session_state.otp_verified = True

                                    st.success("OTP verified successfully!")
                                    st.rerun()

                                else:
                                    record_failed_otp(reset_email)
                                    st.session_state.otp_attempts += 1

                                    is_locked, remaining = get_otp_lock_status(
                                        reset_email
                                    )

                                    if is_locked:
                                        st.session_state.otp_hash = None
                                        st.session_state.otp_expiry = None
                                        st.session_state.otp_sent = False
                                        st.session_state.otp_verified = False

                                        st.error(
                                            "🔒 Five incorrect OTP attempts. "
                                            "Verification is locked for 20 minutes."
                                        )
                                    else:
                                        attempts_left = (
                                            MAX_OTP_ATTEMPTS
                                            - get_otp_failed_count(reset_email)
                                        )
                                        st.error(
                                            "Incorrect OTP. "
                                            f"{attempts_left} attempts remaining."
                                        )


                # --------------------------------------------
                # RESET PASSWORD AFTER OTP VERIFICATION
                # --------------------------------------------

                if st.session_state.otp_verified:

                    st.success(
                        "Email verified. You can reset your password."
                    )

                    new_password = st.text_input(
                        "New password (minimum 8 characters)",
                        type="password",
                        key="otp_new_password"
                    )

                    confirm_password = st.text_input(
                        "Confirm new password",
                        type="password",
                        key="otp_confirm_password"
                    )

                    if st.button(
                        "Reset Password →",
                        use_container_width=True
                    ):

                        if not is_valid_password(new_password):
                            st.error(
                                "Password must contain at least 8 characters "
                                "and must not contain whitespace."
                            )

                        elif new_password != confirm_password:
                            st.error("Passwords do not match.")

                        elif not st.session_state.otp_verified:
                            st.error(
                                "Please verify your email OTP before "
                                "resetting your password."
                            )

                        elif (
                            st.session_state.otp_expiry is None
                            or time.time() > st.session_state.otp_expiry
                        ):
                            st.session_state.otp_hash = None
                            st.session_state.otp_expiry = None
                            st.session_state.otp_sent = False
                            st.session_state.otp_verified = False

                            st.error(
                                "Your OTP verification has expired. "
                                "Please request a new OTP."
                            )

                        elif password_matches_current(
                            reset_email,
                            new_password
                        ):
                            st.error(
                                "You cannot reuse your current password. "
                                "Please choose a different password."
                            )

                        else:
                            with get_db() as conn:
                                cursor = conn.execute(
                                    """
                                    UPDATE users
                                    SET password_hash = ?
                                    WHERE email = ?
                                    """,
                                    (
                                        hash_txt(new_password),
                                        reset_email
                                    )
                                )

                            if cursor.rowcount == 1:
                                clear_reset_state()

                                st.success(
                                    "Password reset successfully! "
                                    "Please sign in with your new password."
                                )

                                time.sleep(1)
                                navigate("Login")
                            else:
                                st.error(
                                    "Password could not be updated. "
                                    "Please restart the password reset process."
                                )


            # ------------------------------------------------
            # CANCEL PASSWORD RESET
            # ------------------------------------------------

            st.markdown("<br>", unsafe_allow_html=True)

            if st.button("← Cancel", use_container_width=True):
                clear_reset_state()
                navigate("Login")


# ============================================================
# DASHBOARDS: ADMIN VS REGULAR USER
# ============================================================

else:

    payload = verify_jwt(st.session_state.token)

    if not payload:
        st.session_state.token = None
        st.session_state.page = "Login"
        st.rerun()

    email = payload["email"]

    with get_db() as conn:
        user_result = conn.execute(
            """
            SELECT username
            FROM users
            WHERE email = ?
            """,
            (email,)
        ).fetchone()

    if not user_result:
        st.session_state.token = None
        st.session_state.page = "Login"
        st.rerun()

    username = user_result[0]

    # ========================================================
    # SIDEBAR MENU
    # ========================================================

    with st.sidebar:

        st.markdown(
            f"""
            <div style="padding:16px 8px;text-align:center;">
                <div style="font-size:28px;">🚢</div>
                <div style="font-weight:700;font-size:16px;color:{COLORS['text_heading']};">
                    Maritime AI
                </div>
                <div style="font-size:11px;color:{COLORS['text_muted']};">
                    {"Admin Panel" if email == "infosys@ai" else "Agentic AI for Maritime Freight Pricing and Route Optimization"}
                </div>
            </div>
            <hr style="border-color:{COLORS['border_light']};">
            """,
            unsafe_allow_html=True
        )

        if email == "infosys@ai":
            options = ["Dashboard", "Settings", "Logout"]
            icons = ["house", "gear", "box-arrow-right"]
        else:
            options = ["Dashboard", "Analytics", "Reports", "Logout"]
            icons = ["house", "graph-up", "file-text", "box-arrow-right"]

        menu = option_menu(
            None,
            options,
            icons=icons,
            styles={
                "container": {
                    "background-color": COLORS["bg_sidebar"]
                },
                "nav-link-selected": {
                    "background-color": COLORS["accent"],
                    "color": COLORS["accent_text"]
                }
            }
        )

        if menu == "Logout":
            st.session_state.token = None
            st.session_state.page = "Login"
            st.rerun()


    # ========================================================
    # ADMIN DASHBOARD
    # ========================================================

    if email == "infosys@ai":

        st.markdown(
            f"""
            <div style="background:{COLORS['text_heading']};border-radius:16px;padding:24px 32px;display:flex;justify-content:space-between;align-items:center;margin-bottom:24px;">
                <div>
                    <h1 style="color:{COLORS['accent']} !important;margin:0;font-size:24px !important;">
                        🚢 Maritime AI
                    </h1>
                    <div style="color:{COLORS['bg_card_alt']};font-size:13px;">
                        Agentic AI for Maritime Freight Pricing and Route Optimization
                    </div>
                </div>
                <div style="background:{COLORS['accent']};padding:8px 18px;border-radius:30px;font-weight:700;color:{COLORS['text_heading']};">
                    🛡️ {username}
                </div>
            </div>
            """,
            unsafe_allow_html=True
        )

        st.markdown(
            f"""
            <div class="pn-card" style="text-align:center;padding:60px 20px;">
                <h1 style="font-size:36px !important;margin-bottom:10px;">
                    🛡️ Admin Dashboard
                </h1>
                <p style="color:{COLORS['text_muted']};font-size:16px;font-weight:500;">
                    Agentic AI for Maritime Freight Pricing and Route Optimization
                </p>
            </div>
            """,
            unsafe_allow_html=True
        )


    # ========================================================
    # REGULAR USER DASHBOARD
    # ========================================================

    else:

        st.markdown(
            f"""
            <div style="background:{COLORS['text_heading']};border-radius:16px;padding:24px 32px;display:flex;justify-content:space-between;align-items:center;margin-bottom:24px;">
                <div>
                    <h1 style="color:{COLORS['accent']} !important;margin:0;font-size:24px !important;">
                        🚢 Maritime AI
                    </h1>
                    <div style="color:{COLORS['bg_card_alt']};font-size:13px;">
                        Agentic AI for Maritime Freight Pricing and Route Optimization
                    </div>
                </div>
                <div style="background:{COLORS['accent']};padding:8px 18px;border-radius:30px;font-weight:700;color:{COLORS['text_heading']};">
                    👤 {username}
                </div>
            </div>
            """,
            unsafe_allow_html=True
        )

        c1, c2, c3, c4 = st.columns(4)

        cards = [
            (c1, "📄", "Documents Indexed", "128"),
            (c2, "🔍", "Searches Today", "47"),
            (c3, "📊", "Efficiency Score", "98.4%"),
            (c4, "🛡️", "Security Status", "Secured")
        ]

        for col, icon, label, value in cards:
            col.markdown(
                f"""
                <div class="pn-card" style="text-align:center;">
                    <div style="font-size:28px;">{icon}</div>
                    <div style="font-size:26px;font-weight:700;color:{COLORS['text_heading']};">
                        {value}
                    </div>
                    <div style="color:{COLORS['text_muted']};font-size:12px;font-weight:600;">
                        {label}
                    </div>
                </div>
                """,
                unsafe_allow_html=True
            )

        st.markdown("<br>", unsafe_allow_html=True)

        fig = go.Figure(
            go.Indicator(
                mode="gauge+number",
                value=92,
                title={
                    "text": "System Health Index",
                    "font": {
                        "color": COLORS["text_heading"],
                        "size": 14
                    }
                },
                gauge={
                    "axis": {"range": [0, 100]},
                    "bar": {"color": COLORS["accent"]},
                    "bgcolor": COLORS["bg_card_alt"],
                    "borderwidth": 1,
                    "bordercolor": COLORS["border"]
                }
            )
        )

        fig.update_layout(
            paper_bgcolor="rgba(0,0,0,0)",
            font={
                "color": COLORS["text_main"],
                "family": "Inter"
            },
            height=260,
            margin=dict(l=10, r=10, t=40, b=10)
        )

        st.plotly_chart(
            fig,
            use_container_width=True
        )

Overwriting app.py


In [9]:
import os
import time
import subprocess

from pyngrok import ngrok
from google.colab import userdata

# ==========================================
# 1. LOAD SECRETS
# ==========================================

NGROK_TOKEN = userdata.get("NGROK_AUTHTOKEN")
GMAIL_ADDRESS = userdata.get("GMAIL_ADDRESS")
EMAIL_PASSWORD = userdata.get("EMAIL_PASSWORD")

if not NGROK_TOKEN:
    raise ValueError(
        "NGROK_AUTHTOKEN is missing from Colab Secrets."
    )

if not GMAIL_ADDRESS or not EMAIL_PASSWORD:
    raise ValueError(
        "GMAIL_ADDRESS or EMAIL_PASSWORD is missing "
        "from Colab Secrets."
    )

# Pass Gmail credentials to the Streamlit subprocess
# through its environment.
os.environ["GMAIL_ADDRESS"] = GMAIL_ADDRESS
os.environ["EMAIL_PASSWORD"] = EMAIL_PASSWORD

# ==========================================
# 2. CONFIGURE NGROK
# ==========================================

from pyngrok import ngrok

# Configure ngrok authentication
try:
    ngrok.set_auth_token(NGROK_TOKEN)
except Exception as e:
    if "Authtoken already set in configuration file" in str(e):
        print("ngrok authentication token is already configured.")
    else:
        raise

# Close existing tunnels.
ngrok.kill()

# ==========================================
# 3. STOP OLD STREAMLIT PROCESSES
# ==========================================

subprocess.run(
    ["pkill", "-f", "streamlit run app.py"],
    check=False
)

time.sleep(2)

# ==========================================
# 4. START STREAMLIT
# ==========================================

process = subprocess.Popen(
    [
        "streamlit",
        "run",
        "app.py",
        "--server.port",
        "8501",
        "--server.headless",
        "true"
    ],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL,
    env=os.environ.copy()
)

# ==========================================
# 5. OPEN NGROK TUNNEL
# ==========================================

try:

    time.sleep(5)

    if process.poll() is not None:
        raise RuntimeError(
            "Streamlit stopped unexpectedly. "
            "Check app.py for errors."
        )

    public_url = ngrok.connect(8501).public_url

    print("=" * 60)
    print("Infosys Portal is running!")
    print(f"Public URL: {public_url}")
    print("=" * 60)

    print("Keep this Colab cell running while using the app.")

    while True:

        if process.poll() is not None:
            print("Streamlit stopped unexpectedly.")
            break

        time.sleep(1)

except KeyboardInterrupt:

    print("Stopping Infosys Portal...")

finally:

    ngrok.kill()

    if process.poll() is None:
        process.terminate()

        try:
            process.wait(timeout=10)
        except subprocess.TimeoutExpired:
            process.kill()

    print("Streamlit and ngrok have been stopped.")

ngrok authentication token is already configured.
Infosys Portal is running!
Public URL: https://gibberish-case-rimmed.ngrok-free.dev
Keep this Colab cell running while using the app.
Stopping Infosys Portal...
Streamlit and ngrok have been stopped.


In [ ]:
%pip install -q pyngrok

In [ ]:
import os
from google.colab import userdata

os.environ["JWT_SECRET"] = userdata.get("JWT_SECRET").strip()

print("JWT_SECRET loaded:", bool(os.environ.get("JWT_SECRET")))

JWT_SECRET loaded: True


In [7]:
from google.colab import files

uploaded = files.upload()

Saving maritime_background.png to maritime_background.png
